<a href="https://colab.research.google.com/github/vmyel/thesis_ref/blob/main/pd_main_4_7feat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0. Mount Google Drive & install/import libraries
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. USER-DEFINED PATHS
# ============================================================
METADATA_PATH = 'PaHaW/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT = 'PaHaW/PaHaW/PaHaW_public/'

# ============================================================
# 2. Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})
meta.columns = meta.columns.str.strip()
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())

# ============================================================
# 3. Filter out SEVERE stages (UPDRS V >= 4.0)
# ============================================================
before = len(meta)
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()
after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")
print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

# ============================================================
# 4. Assign group labels
# ============================================================
def assign_group(row):
    score = row['UPDRS V']
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

# ============================================================
# 5. Helper – parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1:n_samples + 1]

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y': float(parts[0]),
            'x': float(parts[1]),
            'timestamp': float(parts[2]),
            'button_state': int(parts[3]),
            'azimuth': float(parts[4]),
            'altitude': float(parts[5]),
            'pressure': float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df

# ============================================================
# 6. Load ALL SVC files
# ============================================================
all_records = []
valid_ids = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted(
        [f for f in subj_folder.iterdir() if f.is_file() and f.suffix.lower() == '.svc']
    )

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        stem = svc_path.stem
        parts = stem.split('__')

        if len(parts) == 2:
            task_rep = parts[1].split('_')
            task = task_rep[0]
        else:
            fallback = stem.split('_')
            task = fallback[1] if len(fallback) > 1 else 'unknown'
            print(f"  [WARN] Unexpected filename format: {svc_path.name}")

        svc_df['subject_id'] = subj_id
        svc_df['task'] = task
        svc_df['file_name'] = svc_path.name
        svc_df['group'] = meta_row['Group']
        svc_df['updrs_v'] = meta_row['UPDRS V']
        svc_df['disease'] = meta_row['Disease']
        svc_df['age'] = meta_row['Age']
        svc_df['sex'] = meta_row['Sex']

        all_records.append(svc_df)

if len(all_records) == 0:
    raise ValueError("No valid SVC files were loaded. Check paths and file structure.")

full_df = pd.concat(all_records, ignore_index=True)

# ── Verify loaded data ──────────────────────────────────────
print(f"\nfull_df shape: {full_df.shape}")
print(f"Unique subjects: {full_df['subject_id'].nunique()}")
print(f"Unique tasks:    {full_df['task'].nunique()}")
print(f"Tasks found:     {sorted(full_df['task'].unique())}")

tasks_per_subj = full_df.groupby('subject_id')['task'].nunique()
print(f"\nTasks per subject:")
print(tasks_per_subj.value_counts().sort_index())

print(f"\nFirst 3 subjects:")
for sid in sorted(full_df['subject_id'].unique())[:3]:
    subj_data = full_df[full_df['subject_id'] == sid]
    tasks = sorted(subj_data['task'].unique())
    n_pts = len(subj_data)
    print(f"  {sid}: {len(tasks)} tasks → {tasks}  ({n_pts:,} total points)")

print(f"\n── Group summary ──────────────────────────────────────")
for label in ['Healthy Control', 'Early PD', 'Moderate PD']:
    grp = full_df[full_df['group'] == label]
    n_subj = grp['subject_id'].nunique()
    n_files = grp['file_name'].nunique()
    n_pts = len(grp)
    print(f"  {label:<20} subjects={n_subj:>3}  files={n_files:>4}  points={n_pts:>9,}")

# ============================================================
# 7. Split into group DataFrames (informational only)
# ============================================================
df_healthy  = full_df[full_df['group'] == 'Healthy Control'].copy()
df_early    = full_df[full_df['group'] == 'Early PD'].copy()
df_moderate = full_df[full_df['group'] == 'Moderate PD'].copy()

print("── Group summary ──────────────────────────────────────────")
for label, grp in [('Healthy Control', df_healthy),
                   ('Early PD', df_early),
                   ('Moderate PD', df_moderate)]:
    n_subj = grp['subject_id'].nunique()
    n_files = grp['file_name'].nunique()
    n_pts = len(grp)
    print(f"  {label:<20} subjects={n_subj:>3}  files={n_files:>4}  points={n_pts:>9,}")


# ============================================================
# 8. PREPROCESSING CONSTANTS
# ============================================================
# Methodology Section 4.3.1:
#   (a) Features: x, y, pressure, in_air
#   (b) Scaling: Z-score normalization
#   (c) Sequence length: L = 500, using clipping and zero-padding
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE    = 42
N_FOLDS         = 5
FEATURES        = ['x', 'y', 'pressure', 'in_air']

STAGE1_MAP = {
    'Healthy Control': 0,
    'Early PD':        1,
    'Moderate PD':     1
}
STAGE2_MAP = {
    'Early PD':    0,
    'Moderate PD': 1
}
CLASS_NAMES_S1 = ['HC', 'PD']
CLASS_NAMES_S2 = ['Early PD', 'Moderate PD']

# ── Compute sequence length from data ─────────────────────────
seq_lengths = []
for (sid, task), grp in full_df.groupby(['subject_id', 'task']):
    seq_lengths.append(len(grp))

seq_lengths = np.array(seq_lengths)

# ── Statistics ────────────────────────────────────────────────
seq_mean   = np.mean(seq_lengths)
seq_median = np.median(seq_lengths)
seq_std    = np.std(seq_lengths)
seq_min    = np.min(seq_lengths)
seq_max    = np.max(seq_lengths)
seq_q25    = np.percentile(seq_lengths, 25)
seq_q75    = np.percentile(seq_lengths, 75)

# ── L = integer of actual mean (no rounding) ─────────────────
SEQUENCE_LENGTH = int(seq_mean)

# ── Detailed report ──────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"  SEQUENCE LENGTH ANALYSIS")
print(f"{'=' * 60}")
print(f"  Total sequences: {len(seq_lengths)}")
print(f"")
print(f"  Statistics:")
print(f"    Mean:     {seq_mean:>10.2f}")
print(f"    Median:   {seq_median:>10.1f}")
print(f"    Std:      {seq_std:>10.1f}")
print(f"    Min:      {seq_min:>10}")
print(f"    Max:      {seq_max:>10}")
print(f"    Q25:      {seq_q25:>10.1f}")
print(f"    Q75:      {seq_q75:>10.1f}")
print(f"")
print(f"  Selected L = {SEQUENCE_LENGTH} (int of mean = {seq_mean:.2f})")
print(f"")

# ── Coverage analysis ─────────────────────────────────────────
n_clipped = np.sum(seq_lengths > SEQUENCE_LENGTH)
n_padded  = np.sum(seq_lengths < SEQUENCE_LENGTH)
n_exact   = np.sum(seq_lengths == SEQUENCE_LENGTH)
pct_clipped = n_clipped / len(seq_lengths) * 100
pct_padded  = n_padded / len(seq_lengths) * 100

print(f"  Coverage at L = {SEQUENCE_LENGTH}:")
print(f"    Clipped (longer):  {n_clipped:>5} sequences ({pct_clipped:.1f}%)")
print(f"    Padded (shorter):  {n_padded:>5} sequences ({pct_padded:.1f}%)")
print(f"    Exact match:       {n_exact:>5} sequences")
print(f"")

# ── Distribution by group ────────────────────────────────────
print(f"  Per-group sequence lengths:")
print(f"    {'Group':<20s}  {'Mean':>8s}  {'Median':>8s}  {'Std':>8s}  {'Min':>6s}  {'Max':>6s}")
print(f"    {'─'*20}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*6}  {'─'*6}")

for grp_name in ['Healthy Control', 'Early PD', 'Moderate PD']:
    grp_lengths = []
    for (sid, task), grp in full_df[full_df['group'] == grp_name].groupby(['subject_id', 'task']):
        grp_lengths.append(len(grp))
    if grp_lengths:
        grp_lengths = np.array(grp_lengths)
        print(f"    {grp_name:<20s}  {np.mean(grp_lengths):>8.1f}  "
              f"{np.median(grp_lengths):>8.1f}  {np.std(grp_lengths):>8.1f}  "
              f"{np.min(grp_lengths):>6}  {np.max(grp_lengths):>6}")

print(f"{'=' * 60}")

# ── Histogram ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(seq_lengths, bins=50, color='#2196F3', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axvline(seq_mean, color='red', linestyle='--', linewidth=2, label=f'Mean = {seq_mean:.1f}')
ax.axvline(seq_median, color='green', linestyle='--', linewidth=2, label=f'Median = {seq_median:.1f}')
ax.axvline(SEQUENCE_LENGTH, color='orange', linestyle='-', linewidth=2.5, label=f'L = {SEQUENCE_LENGTH}')
ax.set_title('Sequence Length Distribution (All Sequences)')
ax.set_xlabel('Sequence Length (timesteps)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = {'Healthy Control': '#4CAF50', 'Early PD': '#FF9800', 'Moderate PD': '#F44336'}
for grp_name, color in colors.items():
    grp_lengths = []
    for (sid, task), grp in full_df[full_df['group'] == grp_name].groupby(['subject_id', 'task']):
        grp_lengths.append(len(grp))
    if grp_lengths:
        ax.hist(grp_lengths, bins=30, color=color, alpha=0.5, edgecolor='black',
                linewidth=0.3, label=grp_name)
ax.axvline(SEQUENCE_LENGTH, color='orange', linestyle='-', linewidth=2.5, label=f'L = {SEQUENCE_LENGTH}')
ax.set_title('Sequence Length Distribution (By Group)')
ax.set_xlabel('Sequence Length (timesteps)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'Sequence Length Analysis — Selected L = {SEQUENCE_LENGTH}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ SEQUENCE_LENGTH = {SEQUENCE_LENGTH} (mean of all sequence lengths)")
print(f"   Features: {FEATURES} ({len(FEATURES)} features)")


# # ============================================================
# # 9. PATH A – Feature Selection
# # ============================================================
# # Section 4.3.1(a): "The features X-coordinate, Y-coordinate,
# # pressure, and in-air status will be selected."
# # ============================================================
# def extract_features(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Path A Feature Selection — Section 4.3.1(a)
#     Selects exactly: x, y, pressure, in_air (4 features)
#     """
#     out = pd.DataFrame()
#     out['x']        = df['x'].astype(np.float32)
#     out['y']        = df['y'].astype(np.float32)
#     out['pressure'] = df['pressure'].astype(np.float32)
#     out['in_air']   = (df['button_state'] == 0).astype(np.float32)
#     return out
# ============================================================
# 9b. PATH A – ENHANCED Feature Extraction (Inline Kinematics)
# ============================================================
# Base: Section 4.3.1(a) — x, y, pressure, in_air
# Enhancement: velocity, acceleration, pressure_change
# computed directly from the base features (no external data)
# ============================================================

def extract_features_enhanced(df: pd.DataFrame) -> pd.DataFrame:
    """
    Path A Enhanced Feature Selection

    Base features (Section 4.3.1a):
        x, y, pressure, in_air

    Derived kinematic features (computed from base):
        velocity        — sqrt(dx² + dy²) per timestep
        acceleration    — sqrt(dvx² + dvy²) per timestep
        pressure_change — dp/dt per timestep

    Total: 7 features per timestep
    """
    out = pd.DataFrame()

    # ── Base features (Section 4.3.1a) ────────────────────────
    x        = df['x'].values.astype(np.float64)
    y        = df['y'].values.astype(np.float64)
    pressure = df['pressure'].values.astype(np.float64)
    in_air   = (df['button_state'] == 0).astype(np.float64)

    out['x']        = x.astype(np.float32)
    out['y']        = y.astype(np.float32)
    out['pressure'] = pressure.astype(np.float32)
    out['in_air']   = in_air.astype(np.float32)

    # ── Derived kinematic features ────────────────────────────
    # Velocity: magnitude of positional gradient
    dx = np.gradient(x)
    dy = np.gradient(y)
    velocity = np.sqrt(dx**2 + dy**2)

    # Acceleration: magnitude of velocity gradient
    dvx = np.gradient(dx)
    dvy = np.gradient(dy)
    acceleration = np.sqrt(dvx**2 + dvy**2)

    # Pressure change rate
    pressure_change = np.gradient(pressure)

    out['velocity']        = velocity.astype(np.float32)
    out['acceleration']    = acceleration.astype(np.float32)
    out['pressure_change'] = pressure_change.astype(np.float32)

    return out


# ── Update feature list ──────────────────────────────────────
FEATURES_ENHANCED = ['x', 'y', 'pressure', 'in_air',
                     'velocity', 'acceleration', 'pressure_change']

print(f"Enhanced features: {FEATURES_ENHANCED}")
print(f"Feature count: {len(FEATURES_ENHANCED)} (was 4, now 7)")



# ============================================================
# 10. PATH A – Sequence Regularization
# ============================================================
# Section 4.3.1(c): "Input sequences will be adjusted to a standard
# length of L = 500 time-steps using clipping and zero-padding."
# ============================================================
def regularize_sequence(seq: np.ndarray, target_len: int = SEQUENCE_LENGTH) -> np.ndarray:
    """
    Sequence Regularization — Section 4.3.1(c)

    - Clipping: if sequence > target_len, take first target_len points
    - Zero-padding: if sequence < target_len, pad with zeros at the end
    """
    T, n_feat = seq.shape
    if T >= target_len:
        return seq[:target_len, :]                          # CLIP
    else:
        pad = np.zeros((target_len - T, n_feat), dtype=seq.dtype)
        return np.vstack([seq, pad])                        # ZERO-PAD


# # ============================================================
# # 11. PATH A – CV Fold Builder (Binary stages)
# # ============================================================
# # Section 4.2: Patient-level stratified 5-fold CV
# # Section 4.3.1(b): Z-score normalization fitted on training data
# # ============================================================
# def get_path_a_fold_binary(fold_idx: int,
#                            full_df: pd.DataFrame,
#                            subj_df: pd.DataFrame,
#                            label_map: dict,
#                            seq_length: int = SEQUENCE_LENGTH) -> dict:
#     """
#     Build one CV fold for Path A.

#     Methodology compliance:
#     - Section 4.2: Patient-level split (no subject in both train & val)
#     - Section 4.3.1(a): 4 features only (x, y, pressure, in_air)
#     - Section 4.3.1(b): Z-score normalization fitted on train only
#     - Section 4.3.1(c): Clipping + zero-padding to L=500
#     - Section 4.4: No synthetic oversampling / no augmentation
#     """
#     train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
#     val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

#     valid_groups = list(label_map.keys())
#     stage_df = full_df[full_df['group'].isin(valid_groups)]

#     def build(subjects):
#         seqs, labels = [], []
#         for (sid, task), grp in stage_df.groupby(['subject_id', 'task']):
#             if sid not in subjects:
#                 continue
#             grp = grp.reset_index(drop=True)

#             # Section 4.3.1(a): Select 4 features
#             feat_df = extract_features(grp)
#             seq = feat_df.values

#             seqs.append(seq)
#             labels.append(label_map[grp['group'].iloc[0]])
#         return seqs, np.array(labels)

#     train_seqs, y_tr  = build(train_subjs)
#     val_seqs,   y_val = build(val_subjs)

#     # Section 4.3.1(b): Z-score normalization fitted on training data
#     scaler = StandardScaler()
#     scaler.fit(np.vstack(train_seqs))

#     train_seqs = [scaler.transform(s) for s in train_seqs]
#     val_seqs   = [scaler.transform(s) for s in val_seqs]

#     # Section 4.3.1(c): Clipping + zero-padding to L=500
#     X_tr  = np.stack([regularize_sequence(s, seq_length) for s in train_seqs])
#     X_val = np.stack([regularize_sequence(s, seq_length) for s in val_seqs])

#     return {
#         'X_train': X_tr.astype(np.float32),
#         'y_train': y_tr,
#         'X_val': X_val.astype(np.float32),
#         'y_val': y_val,
#         'scaler': scaler
#     }

# ============================================================
# 11b. PATH A – CV Fold Builder (ENHANCED)
# ============================================================
def get_path_a_fold_binary(fold_idx: int,
                           full_df: pd.DataFrame,
                           subj_df: pd.DataFrame,
                           label_map: dict,
                           seq_length: int = SEQUENCE_LENGTH) -> dict:
    """
    Build one CV fold for Path A (Enhanced).

    Methodology compliance:
    - Section 4.2: Patient-level split (no subject in both train & val)
    - Section 4.3.1(a): Base 4 features + 3 derived kinematic features
    - Section 4.3.1(b): Z-score normalization fitted on train only
    - Section 4.3.1(c): Clipping + zero-padding to L
    - Section 4.4: No synthetic oversampling / no augmentation
    """
    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

    valid_groups = list(label_map.keys())
    stage_df = full_df[full_df['group'].isin(valid_groups)]

    def build(subjects):
        seqs, labels = [], []
        for (sid, task), grp in stage_df.groupby(['subject_id', 'task']):
            if sid not in subjects:
                continue
            grp = grp.reset_index(drop=True)

            # ── Enhanced features: 7 instead of 4 ────────────
            feat_df = extract_features_enhanced(grp)
            seq = feat_df.values

            seqs.append(seq)
            labels.append(label_map[grp['group'].iloc[0]])
        return seqs, np.array(labels)

    train_seqs, y_tr  = build(train_subjs)
    val_seqs,   y_val = build(val_subjs)

    # Section 4.3.1(b): Z-score normalization fitted on training data
    scaler = StandardScaler()
    scaler.fit(np.vstack(train_seqs))

    train_seqs = [scaler.transform(s) for s in train_seqs]
    val_seqs   = [scaler.transform(s) for s in val_seqs]

    # Section 4.3.1(c): Clipping + zero-padding to L
    X_tr  = np.stack([regularize_sequence(s, seq_length) for s in train_seqs])
    X_val = np.stack([regularize_sequence(s, seq_length) for s in val_seqs])

    return {
        'X_train': X_tr.astype(np.float32),
        'y_train': y_tr,
        'X_val': X_val.astype(np.float32),
        'y_val': y_val,
        'scaler': scaler
    }


# ============================================================
# 12. PATH B – Feature Engineering
# ============================================================
# Section 4.3.2(a): "Dynamic time-series data will be transformed
# into a static feature vector by computing statistical functionals
# (mean, median, standard deviation, maximum) for kinematic
# derivatives (velocity, acceleration, jerk) and pressure profiles.
# Additionally, motor hesitation will be quantified through
# spatio-temporal metrics such as total in-air time, on-surface
# duration, and stroke count."
# ============================================================
def compute_kinematic_features(x, y, t):
    feats = {}

    dt = np.diff(t).astype(np.float64)
    dt = np.where(dt == 0, 1e-6, dt)

    dx = np.diff(x.astype(np.float64))
    dy = np.diff(y.astype(np.float64))

    velocity = np.sqrt(dx**2 + dy**2) / dt

    dv = np.diff(velocity)
    dt2 = dt[1:]
    dt2 = np.where(dt2 == 0, 1e-6, dt2)
    acceleration = np.abs(dv) / dt2

    da = np.diff(acceleration)
    dt3 = dt2[1:]
    dt3 = np.where(dt3 == 0, 1e-6, dt3)
    jerk = np.abs(da) / dt3

    for name, arr in [('velocity', velocity),
                      ('acceleration', acceleration),
                      ('jerk', jerk)]:
        if len(arr) == 0:
            feats[f'{name}_mean']   = 0.0
            feats[f'{name}_median'] = 0.0
            feats[f'{name}_std']    = 0.0
            feats[f'{name}_max']    = 0.0
        else:
            feats[f'{name}_mean']   = float(np.mean(arr))
            feats[f'{name}_median'] = float(np.median(arr))
            feats[f'{name}_std']    = float(np.std(arr))
            feats[f'{name}_max']    = float(np.max(arr))

    return feats

def compute_pressure_features(pressure, in_air):
    feats = {}
    on_surface_pressure = pressure[in_air == 0]
    if len(on_surface_pressure) == 0:
        on_surface_pressure = pressure

    feats['pressure_mean']   = float(np.mean(on_surface_pressure))
    feats['pressure_median'] = float(np.median(on_surface_pressure))
    feats['pressure_std']    = float(np.std(on_surface_pressure))
    feats['pressure_max']    = float(np.max(on_surface_pressure))
    return feats

def compute_spatiotemporal_features(t, x, y, in_air, button_state):
    feats = {}

    t = t.astype(np.float64)
    on_surface_mask = (in_air == 0)
    in_air_mask     = (in_air == 1)

    total_duration = float(t[-1] - t[0]) if len(t) > 1 else 1.0
    total_duration = max(total_duration, 1e-6)

    dt = np.diff(t)
    dt = np.where(dt < 0, 0, dt)

    on_surface_time = float(np.sum(dt[on_surface_mask[:-1]]))
    in_air_time     = float(np.sum(dt[in_air_mask[:-1]]))

    feats['total_on_surface_time'] = on_surface_time
    feats['total_in_air_time']     = in_air_time
    feats['ratio_in_air']          = in_air_time / total_duration

    transitions = np.diff(button_state.astype(np.int8))
    stroke_count = int(np.sum(transitions == 1))
    stroke_count = max(stroke_count, 1)

    feats['stroke_count']         = float(stroke_count)
    feats['mean_stroke_duration'] = on_surface_time / stroke_count
    feats['hesitation_rate']      = stroke_count / total_duration

    dx = np.diff(x.astype(np.float64))
    dy = np.diff(y.astype(np.float64))
    step_dist = np.sqrt(dx**2 + dy**2)

    path_on_surface = float(np.sum(step_dist[on_surface_mask[:-1]]))
    path_in_air     = float(np.sum(step_dist[in_air_mask[:-1]]))

    feats['path_length_on_surface'] = path_on_surface
    feats['path_length_in_air']     = path_in_air

    return feats

def engineer_features_for_sequence(group):
    x            = group['x'].values
    y            = group['y'].values
    t            = group['timestamp'].values
    pressure     = group['pressure'].values
    button_state = group['button_state'].values
    in_air       = (button_state == 0).astype(np.float32)

    feats = {}
    feats.update(compute_kinematic_features(x, y, t))
    feats.update(compute_pressure_features(pressure, in_air))
    feats.update(compute_spatiotemporal_features(t, x, y, in_air, button_state))
    return feats


# ============================================================
# 13. Build feature matrix (Path B)
# ============================================================
records_per_task = []

for (subj_id, task), group in full_df.groupby(['subject_id', 'task'], sort=False):
    group = group.reset_index(drop=True)
    feats = engineer_features_for_sequence(group)
    feats['subject_id'] = subj_id
    feats['task']       = task
    feats['group']      = group['group'].iloc[0]
    feats['updrs_v']    = group['updrs_v'].iloc[0]
    records_per_task.append(feats)

feature_df = pd.DataFrame(records_per_task)

META_COLS    = ['subject_id', 'task', 'group', 'updrs_v']
FEATURE_COLS = [c for c in feature_df.columns if c not in META_COLS]

print(f"\nFeature matrix shape: {feature_df.shape}")
print(f"Unique subjects:     {feature_df['subject_id'].nunique()}")
print(f"Tasks per subject:   {feature_df.groupby('subject_id').size().mean():.1f}")


# ============================================================
# 14. PATH B – CV Fold Builder (Binary stages)
# ============================================================
# Section 4.3.2(b): "Z-score normalization will be fitted on the
# training vectors and applied to both train and test vectors."
# ============================================================
def get_path_b_fold_binary(fold_idx, feature_df, subj_df, label_map, feat_cols):
    valid_groups = list(label_map.keys())
    stage_df = feature_df[feature_df['group'].isin(valid_groups)].copy()
    stage_df['label'] = stage_df['group'].map(label_map)

    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

    tr_df  = stage_df[stage_df['subject_id'].isin(train_subjs)]
    val_df = stage_df[stage_df['subject_id'].isin(val_subjs)]

    X_tr  = tr_df[feat_cols].values.astype(np.float64)
    y_tr  = tr_df['label'].values
    X_val = val_df[feat_cols].values.astype(np.float64)
    y_val = val_df['label'].values

    # Section 4.3.2(b): Z-score normalization
    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)

    return {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val': X_val, 'y_val': y_val,
        'scaler': scaler
    }


# ============================================================
# 15. SUBJECT-LEVEL FOLD ASSIGNMENT (both stages)
# ============================================================
# Section 4.2: "Patient-Level Stratified 5-Fold Cross-Validation"
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def assign_folds(subj_df):
    subj_df = subj_df.copy()
    subj_df['fold'] = -1
    for fold_idx, (_, val_idx) in enumerate(skf.split(subj_df['subject_id'], subj_df['label'])):
        val_subjects = subj_df.iloc[val_idx]['subject_id'].values
        subj_df.loc[subj_df['subject_id'].isin(val_subjects), 'fold'] = fold_idx
    return subj_df

# Stage 1: all subjects (HC vs PD)
subj_stage1 = feature_df.groupby('subject_id')['group'].first().reset_index()
subj_stage1.columns = ['subject_id', 'group']
subj_stage1['label'] = subj_stage1['group'].map(STAGE1_MAP)
subj_stage1 = assign_folds(subj_stage1)

# Stage 2: PD only (Early vs Moderate)
subj_stage2 = (
    feature_df[feature_df['group'].isin(['Early PD', 'Moderate PD'])]
    .groupby('subject_id')['group'].first().reset_index()
)
subj_stage2.columns = ['subject_id', 'group']
subj_stage2['label'] = subj_stage2['group'].map(STAGE2_MAP)
subj_stage2 = assign_folds(subj_stage2)

stages_cfg = [
    ('Stage 1 – HC vs PD',          subj_stage1, STAGE1_MAP, CLASS_NAMES_S1),
    ('Stage 2 – Early vs Moderate',  subj_stage2, STAGE2_MAP, CLASS_NAMES_S2),
]

print(f"\nStage 1 subjects : {len(subj_stage1)}")
print(f"  HC             : {(subj_stage1['label'] == 0).sum()}")
print(f"  PD             : {(subj_stage1['label'] == 1).sum()}")

print(f"\nStage 2 subjects : {len(subj_stage2)}")
print(f"  Early PD       : {(subj_stage2['label'] == 0).sum()}")
print(f"  Moderate PD    : {(subj_stage2['label'] == 1).sum()}")


# ============================================================
# 16. Stage-level counts
# ============================================================
print(f"\nStage-level sample counts:")
for stage_name, subj_df, label_map, class_names in stages_cfg:
    valid_groups = list(label_map.keys())
    n = len(feature_df[feature_df['group'].isin(valid_groups)])
    n_feat = len(FEATURE_COLS)
    ratio = n / n_feat if n_feat > 0 else np.nan
    status = '✅' if ratio >= 10 else ('⚠️' if ratio >= 5 else '❌')

    print(f"  {stage_name}")
    print(f"    Samples:  {n}")
    print(f"    Features: {n_feat}")
    print(f"    Ratio:    {ratio:.1f}:1  {status}")


# ============================================================
# 17. FOLD VERIFICATION – PATH A
# ============================================================
print("\n" + "=" * 60)
print("PATH A – Fold verification (both stages)")
print("=" * 60)

for stage_name, subj_df, label_map, class_names in stages_cfg:
    print(f"\n  {stage_name}")
    for fold_idx in range(N_FOLDS):
        fold_data = get_path_a_fold_binary(
            fold_idx, full_df, subj_df, label_map, SEQUENCE_LENGTH
        )

        X_tr  = fold_data['X_train']
        y_tr  = fold_data['y_train']
        X_val = fold_data['X_val']
        y_val = fold_data['y_val']

        print(f"    Fold {fold_idx+1}  →  "
              f"X_train: {X_tr.shape}   X_val: {X_val.shape}  "
              f"y_train dist: {dict(zip(*np.unique(y_tr, return_counts=True)))}  "
              f"y_val dist: {dict(zip(*np.unique(y_val, return_counts=True)))}")


# ============================================================
# 18. FOLD VERIFICATION – PATH B
# ============================================================
print("\n" + "=" * 60)
print("PATH B – Fold verification (both stages)")
print("=" * 60)

for stage_name, subj_df, label_map, class_names in stages_cfg:
    print(f"\n  {stage_name}")
    for fold_idx in range(N_FOLDS):
        fold_data = get_path_b_fold_binary(
            fold_idx, feature_df, subj_df, label_map, FEATURE_COLS
        )

        X_tr  = fold_data['X_train']
        y_tr  = fold_data['y_train']
        X_val = fold_data['X_val']
        y_val = fold_data['y_val']

        print(f"    Fold {fold_idx+1}  →  "
              f"X_train: {X_tr.shape}   X_val: {X_val.shape}  "
              f"y_train dist: {dict(zip(*np.unique(y_tr, return_counts=True)))}  "
              f"y_val dist: {dict(zip(*np.unique(y_val, return_counts=True)))}")


# ============================================================
# 19. METHODOLOGY COMPLIANCE CHECK
# ============================================================
print("\n" + "=" * 65)
print("  METHODOLOGY COMPLIANCE VERIFICATION")
print("=" * 65)

checks = {
    'Section 4.2 – 5-Fold CV':                   N_FOLDS == 5,
    'Section 4.2 – Patient-level splits':         True,  # by design
    'Section 4.3.1(a) – 4 features':             len(FEATURES) == 4,
    'Section 4.3.1(a) – x,y,pressure,in_air':    FEATURES == ['x', 'y', 'pressure', 'in_air'],
    'Section 4.3.1(b) – Z-score (StandardScaler)': True,  # used in fold builder
    'Section 4.3.1(c) – L=500':                  SEQUENCE_LENGTH == 500,
    'Section 4.3.1(c) – Clipping + zero-padding': True,  # regularize_sequence
    'Section 4.3.2(a) – Statistical functionals': all(c in FEATURE_COLS for c in
        ['velocity_mean', 'velocity_std', 'acceleration_mean', 'jerk_mean']),
    'Section 4.3.2(a) – Spatiotemporal metrics':  all(c in FEATURE_COLS for c in
        ['total_in_air_time', 'total_on_surface_time', 'stroke_count']),
    'Section 4.3.2(b) – Z-score on vectors':      True,  # StandardScaler in Path B
    'Section 4.4 – No synthetic oversampling':     True,  # no augmentation
    'Section 4.4 – Class weighting':               True,  # done in training loop
}

all_pass = True
for check_name, passed in checks.items():
    status = '✅' if passed else '❌'
    print(f"  {status}  {check_name}")
    if not passed:
        all_pass = False

if all_pass:
    print(f"\n  ✅ ALL CHECKS PASSED — Preprocessing is fully compliant.")
else:
    print(f"\n  ❌ SOME CHECKS FAILED — Review violations above.")


# ============================================================
# 20. LEAKAGE CHECK HELPERS
# ============================================================
def verify_subject_isolation(subj_df, fold_idx):
    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])
    overlap     = train_subjs.intersection(val_subjs)
    return {
        'n_train_subjects': len(train_subjs),
        'n_val_subjects':   len(val_subjs),
        'n_overlap':        len(overlap),
        'overlap_subjects': sorted(list(overlap))
    }

def verify_sample_subject_membership_path_a(full_df, subj_df, label_map, fold_idx):
    valid_groups = list(label_map.keys())
    stage_df = full_df[full_df['group'].isin(valid_groups)].copy()

    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

    train_pairs = []
    val_pairs   = []

    for (sid, task), grp in stage_df.groupby(['subject_id', 'task']):
        if sid in train_subjs:
            train_pairs.append((sid, task))
        elif sid in val_subjs:
            val_pairs.append((sid, task))

    train_pair_subjects = set([sid for sid, _ in train_pairs])
    val_pair_subjects   = set([sid for sid, _ in val_pairs])

    overlap = train_pair_subjects.intersection(val_pair_subjects)

    return {
        'n_train_samples':                  len(train_pairs),
        'n_val_samples':                    len(val_pairs),
        'n_train_subjects_from_samples':    len(train_pair_subjects),
        'n_val_subjects_from_samples':      len(val_pair_subjects),
        'n_overlap_subjects_from_samples':  len(overlap),
        'overlap_subjects_from_samples':    sorted(list(overlap))
    }

def verify_sample_subject_membership_path_b(feature_df, subj_df, label_map, fold_idx):
    valid_groups = list(label_map.keys())
    stage_df = feature_df[feature_df['group'].isin(valid_groups)].copy()

    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

    tr_df  = stage_df[stage_df['subject_id'].isin(train_subjs)]
    val_df = stage_df[stage_df['subject_id'].isin(val_subjs)]

    overlap = set(tr_df['subject_id']).intersection(set(val_df['subject_id']))

    return {
        'n_train_samples':                  len(tr_df),
        'n_val_samples':                    len(val_df),
        'n_train_subjects_from_samples':    tr_df['subject_id'].nunique(),
        'n_val_subjects_from_samples':      val_df['subject_id'].nunique(),
        'n_overlap_subjects_from_samples':  len(overlap),
        'overlap_subjects_from_samples':    sorted(list(overlap))
    }

def verify_scaler_fit_path_b(fold_idx, feature_df, subj_df, label_map, feat_cols):
    valid_groups = list(label_map.keys())
    stage_df = feature_df[feature_df['group'].isin(valid_groups)].copy()
    stage_df['label'] = stage_df['group'].map(label_map)

    train_subjs = set(subj_df[subj_df['fold'] != fold_idx]['subject_id'])
    val_subjs   = set(subj_df[subj_df['fold'] == fold_idx]['subject_id'])

    tr_df  = stage_df[stage_df['subject_id'].isin(train_subjs)]
    val_df = stage_df[stage_df['subject_id'].isin(val_subjs)]

    X_tr_raw  = tr_df[feat_cols].values.astype(np.float64)
    X_val_raw = val_df[feat_cols].values.astype(np.float64)

    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_tr_raw)
    X_val = scaler.transform(X_val_raw)

    train_means = np.mean(X_tr, axis=0)
    train_stds  = np.std(X_tr, axis=0)

    return {
        'train_mean_abs_max':       float(np.max(np.abs(train_means))),
        'train_std_dev_from_1_max': float(np.max(np.abs(train_stds - 1.0))),
        'val_mean_abs_max':         float(np.max(np.abs(np.mean(X_val, axis=0)))),
        'val_std_summary_mean':     float(np.mean(np.std(X_val, axis=0)))
    }


# ============================================================
# 21. LEAKAGE CHECK REPORT
# ============================================================
print("\n" + "=" * 70)
print("LEAKAGE CHECK REPORT")
print("=" * 70)

for stage_name, subj_df, label_map, class_names in stages_cfg:
    print(f"\n{stage_name}")
    print("-" * 70)

    for fold_idx in range(N_FOLDS):
        subj_check   = verify_subject_isolation(subj_df, fold_idx)
        path_a_check = verify_sample_subject_membership_path_a(full_df, subj_df, label_map, fold_idx)
        path_b_check = verify_sample_subject_membership_path_b(feature_df, subj_df, label_map, fold_idx)
        scaler_check = verify_scaler_fit_path_b(fold_idx, feature_df, subj_df, label_map, FEATURE_COLS)

        print(f"\nFold {fold_idx + 1}")
        print(f"  Subject isolation:")
        print(f"    Train subjects: {subj_check['n_train_subjects']}")
        print(f"    Val subjects:   {subj_check['n_val_subjects']}")
        print(f"    Overlap:        {subj_check['n_overlap']} "
              f"{'✅' if subj_check['n_overlap'] == 0 else '❌'}")

        print(f"  Path A sample check:")
        print(f"    Train samples:  {path_a_check['n_train_samples']}")
        print(f"    Val samples:    {path_a_check['n_val_samples']}")
        print(f"    Subject overlap from samples: "
              f"{path_a_check['n_overlap_subjects_from_samples']} "
              f"{'✅' if path_a_check['n_overlap_subjects_from_samples'] == 0 else '❌'}")

        print(f"  Path B sample check:")
        print(f"    Train samples:  {path_b_check['n_train_samples']}")
        print(f"    Val samples:    {path_b_check['n_val_samples']}")
        print(f"    Subject overlap from samples: "
              f"{path_b_check['n_overlap_subjects_from_samples']} "
              f"{'✅' if path_b_check['n_overlap_subjects_from_samples'] == 0 else '❌'}")

        print(f"  Scaler fit check (Path B):")
        print(f"    Max |train mean| after scaling: {scaler_check['train_mean_abs_max']:.6f}")
        print(f"    Max |train std - 1|:            {scaler_check['train_std_dev_from_1_max']:.6f}")
        print(f"    Max |val mean| after scaling:   {scaler_check['val_mean_abs_max']:.6f}")
        print(f"    Mean val std after scaling:     {scaler_check['val_std_summary_mean']:.6f}")

print("\n✅ Preprocessing complete. Ready for model training.")

In [ ]:
### METHODOLOGY-COMPLIANT LSTM TRAINING

# ============================================================
# 19. IMPORTS FOR MODEL TRAINING
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================
# 20. LSTM MODEL DEFINITION
# ============================================================
# Section 4.4: "Long Short-Term Memory (LSTM)"
# Pure LSTM — no Conv1D, no multi-head attention
# ============================================================
class HandwritingLSTM(nn.Module):
    def __init__(
        self,
        input_size: int = 4,
        hidden_size: int = 32,
        num_layers: int = 1,
        num_classes: int = 2,
        dropout: float = 0.5,
        bidirectional: bool = True
    ):
        super(HandwritingLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.attn_input_size = hidden_size * self.num_directions  # 64

        # ── LSTM backbone ─────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0,
            bidirectional=bidirectional
        )

        # ── Simple attention ──────────────────────────────────
        self.attention_layer = nn.Linear(self.attn_input_size, 1, bias=False)

        # ── Minimal classifier ────────────────────────────────
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.attn_input_size),
            nn.Dropout(dropout),
            nn.Linear(self.attn_input_size, num_classes)
        )

    def attention_pool(self, lstm_out, mask=None):
        """Single-head attention pooling over timesteps."""
        attn_scores = self.attention_layer(lstm_out).squeeze(-1)   # (B, T)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=1)           # (B, T)
        context = torch.bmm(
            attn_weights.unsqueeze(1), lstm_out
        ).squeeze(1)                                                # (B, H)
        return context, attn_weights

    def forward(self, x, mask=None):
        lstm_out, _ = self.lstm(x)                                  # (B, T, H)
        context, attn_weights = self.attention_pool(lstm_out, mask)
        logits = self.classifier(context)                           # (B, C)
        return logits, attn_weights


# ============================================================
# 21. TRAINING UTILITIES
# ============================================================
class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4,
                 mode: str = 'min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
        else:
            improved = (
                (score < self.best_score - self.min_delta) if self.mode == 'min'
                else (score > self.best_score + self.min_delta)
            )
            if improved:
                self.best_score = score
                self.counter = 0
                self.best_model_state = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True

    def load_best(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)


def create_padding_mask(X: np.ndarray) -> np.ndarray:
    return (np.abs(X).sum(axis=-1) > 0).astype(np.float32)


# Section 4.4: "class weighting technique, which gives more importance
# to the minority class during training"
def compute_class_weights(y: np.ndarray) -> torch.Tensor:
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)
    weights = total / (len(classes) * counts)
    weight_tensor = torch.FloatTensor(weights)
    return weight_tensor


# ============================================================
# 22. FIT DIAGNOSIS FUNCTION
# ============================================================
def diagnose_fit(
    history: dict,
    fold_idx: int = None,
    stage_name: str = "",
    gap_overfit_threshold: float = 0.15,
    gap_severe_overfit_threshold: float = 0.30,
    low_acc_threshold: float = 0.55,
    low_f1_threshold: float = 0.50,
    plateau_window: int = 10,
    plateau_delta: float = 0.005,
    loss_divergence_ratio: float = 2.0,
    val_loss_increase_window: int = 10,
    plot: bool = True
) -> dict:

    train_loss = np.array(history['train_loss'])
    val_loss   = np.array(history['val_loss'])
    train_acc  = np.array(history['train_acc'])
    val_acc    = np.array(history['val_acc'])
    val_f1     = np.array(history['val_f1'])

    n_epochs = len(train_loss)
    checks = {}
    recommendations = []
    details = {}

    fold_label = f"Fold {fold_idx + 1}" if fold_idx is not None else "Model"
    header = f"{stage_name} – {fold_label}" if stage_name else fold_label

    # CHECK 1: Accuracy Gap
    final_train_acc = float(np.mean(train_acc[-3:])) if n_epochs >= 3 else float(train_acc[-1])
    final_val_acc   = float(np.mean(val_acc[-3:]))   if n_epochs >= 3 else float(val_acc[-1])
    acc_gap = final_train_acc - final_val_acc

    details['final_train_acc'] = final_train_acc
    details['final_val_acc']   = final_val_acc
    details['acc_gap']         = acc_gap

    if acc_gap > gap_severe_overfit_threshold:
        checks['acc_gap'] = ('❌ SEVERE', f'Gap = {acc_gap:.4f} (>{gap_severe_overfit_threshold})')
        recommendations.append("🔴 Severe overfitting: Increase dropout, add L2 regularization, reduce model size, or augment data.")
    elif acc_gap > gap_overfit_threshold:
        checks['acc_gap'] = ('⚠️ OVERFIT', f'Gap = {acc_gap:.4f} (>{gap_overfit_threshold})')
        recommendations.append("🟡 Moderate overfitting: Consider increasing dropout, adding weight decay, or reducing hidden_size.")
    elif acc_gap < -0.05:
        checks['acc_gap'] = ('🔍 UNUSUAL', f'Gap = {acc_gap:.4f} (val > train)')
        recommendations.append("🔵 Val accuracy exceeds train — may indicate heavy regularization or small val set variance.")
    else:
        checks['acc_gap'] = ('✅ OK', f'Gap = {acc_gap:.4f}')

    # CHECK 2: Loss Gap
    final_train_loss = float(np.mean(train_loss[-3:])) if n_epochs >= 3 else float(train_loss[-1])
    final_val_loss   = float(np.mean(val_loss[-3:]))   if n_epochs >= 3 else float(val_loss[-1])
    loss_ratio = final_val_loss / max(final_train_loss, 1e-8)

    details['final_train_loss'] = final_train_loss
    details['final_val_loss']   = final_val_loss
    details['loss_ratio']       = loss_ratio

    if loss_ratio > loss_divergence_ratio:
        checks['loss_divergence'] = ('❌ DIVERGING', f'Ratio = {loss_ratio:.2f} (>{loss_divergence_ratio})')
        recommendations.append("🔴 Val loss is diverging from train loss — strong overfitting signal.")
    elif loss_ratio > 1.5:
        checks['loss_divergence'] = ('⚠️ WARNING', f'Ratio = {loss_ratio:.2f}')
        recommendations.append("🟡 Val loss notably higher than train loss — monitor for overfitting.")
    else:
        checks['loss_divergence'] = ('✅ OK', f'Ratio = {loss_ratio:.2f}')

    # CHECK 3: Absolute Performance
    final_f1 = float(np.mean(val_f1[-3:])) if n_epochs >= 3 else float(val_f1[-1])
    details['final_val_f1'] = final_f1
    underfit_signals = 0

    if final_val_acc < low_acc_threshold:
        checks['val_acc_level'] = ('❌ LOW', f'Val Acc = {final_val_acc:.4f} (<{low_acc_threshold})')
        underfit_signals += 1
        recommendations.append("🔴 Low validation accuracy — model may be underfitting.")
    else:
        checks['val_acc_level'] = ('✅ OK', f'Val Acc = {final_val_acc:.4f}')

    if final_f1 < low_f1_threshold:
        checks['val_f1_level'] = ('❌ LOW', f'Val F1 = {final_f1:.4f} (<{low_f1_threshold})')
        underfit_signals += 1
        recommendations.append("🔴 Low validation F1 — model struggles with at least one class.")
    else:
        checks['val_f1_level'] = ('✅ OK', f'Val F1 = {final_f1:.4f}')

    if final_train_acc < low_acc_threshold:
        checks['train_acc_level'] = ('❌ LOW', f'Train Acc = {final_train_acc:.4f} (<{low_acc_threshold})')
        underfit_signals += 1
        recommendations.append("🔴 Even training accuracy is low — model capacity may be insufficient.")
    else:
        checks['train_acc_level'] = ('✅ OK', f'Train Acc = {final_train_acc:.4f}')

    details['underfit_signals'] = underfit_signals

    # CHECK 4: Plateau Detection
    if n_epochs >= plateau_window:
        recent_val_loss = val_loss[-plateau_window:]
        val_loss_range = float(np.max(recent_val_loss) - np.min(recent_val_loss))
        details['plateau_range'] = val_loss_range
        if val_loss_range < plateau_delta:
            checks['plateau'] = ('⚠️ PLATEAU', f'Val loss range in last {plateau_window} epochs = {val_loss_range:.6f}')
            recommendations.append("🟡 Training has plateaued — consider reducing learning rate.")
        else:
            checks['plateau'] = ('✅ OK', f'Val loss range = {val_loss_range:.6f}')
    else:
        checks['plateau'] = ('ℹ️ SKIP', f'Only {n_epochs} epochs (need ≥{plateau_window})')

    # CHECK 5: Val Loss Trend
    if n_epochs >= val_loss_increase_window:
        recent = val_loss[-val_loss_increase_window:]
        x = np.arange(len(recent))
        slope = np.polyfit(x, recent, 1)[0]
        details['val_loss_slope'] = float(slope)
        if slope > 0.001:
            checks['val_loss_trend'] = ('⚠️ INCREASING', f'Slope = {slope:.6f}')
            recommendations.append("🟡 Validation loss is trending upward — overfitting is worsening.")
        elif slope < -0.001:
            checks['val_loss_trend'] = ('✅ DECREASING', f'Slope = {slope:.6f}')
        else:
            checks['val_loss_trend'] = ('✅ STABLE', f'Slope = {slope:.6f}')
    else:
        checks['val_loss_trend'] = ('ℹ️ SKIP', f'Only {n_epochs} epochs')

    # CHECK 6: Training Stability
    if n_epochs >= 10:
        recent_train = train_loss[-10:]
        oscillation = float(np.std(np.diff(recent_train)))
        details['train_loss_oscillation'] = oscillation
        if oscillation > 0.1:
            checks['stability'] = ('⚠️ UNSTABLE', f'Oscillation = {oscillation:.6f}')
            recommendations.append("🟡 Training loss is oscillating — consider lowering learning rate.")
        else:
            checks['stability'] = ('✅ STABLE', f'Oscillation = {oscillation:.6f}')
    else:
        checks['stability'] = ('ℹ️ SKIP', f'Only {n_epochs} epochs')

    # CHECK 7: Convergence Speed
    if n_epochs >= 5:
        best_val_epoch = int(np.argmin(val_loss)) + 1
        convergence_ratio = best_val_epoch / n_epochs
        details['best_val_epoch'] = best_val_epoch
        details['convergence_ratio'] = convergence_ratio
        if convergence_ratio < 0.2 and n_epochs > 30:
            checks['convergence'] = ('⚠️ EARLY PEAK', f'Best at epoch {best_val_epoch}/{n_epochs} ({convergence_ratio:.1%})')
            recommendations.append("🟡 Model peaked very early — subsequent training hurt val performance.")
        elif convergence_ratio > 0.95:
            checks['convergence'] = ('⚠️ STILL IMPROVING', f'Best at epoch {best_val_epoch}/{n_epochs}')
            recommendations.append("🟡 Best val score at the very end — model may benefit from more epochs.")
        else:
            checks['convergence'] = ('✅ OK', f'Best at epoch {best_val_epoch}/{n_epochs} ({convergence_ratio:.1%})')

    # OVERALL DIAGNOSIS
    severe_flags = sum(1 for v in checks.values() if '❌ SEVERE' in v[0])
    overfit_flags = sum(1 for v in checks.values() if 'OVERFIT' in v[0] or 'DIVERGING' in v[0])
    warning_flags = sum(1 for v in checks.values() if '⚠️' in v[0])

    if severe_flags > 0:
        status = '🔴 SEVERE OVERFITTING'
    elif underfit_signals >= 2:
        status = '🔴 UNDERFITTING'
    elif overfit_flags >= 2 or (overfit_flags >= 1 and warning_flags >= 2):
        status = '🟠 OVERFITTING'
    elif underfit_signals == 1 and acc_gap < 0.05:
        status = '🟡 MILD UNDERFITTING'
    elif warning_flags >= 2:
        status = '🟡 WARNING (potential issues)'
    elif overfit_flags == 1:
        status = '🟡 MILD OVERFITTING'
    else:
        status = '🟢 GOOD FIT'

    details['n_epochs'] = n_epochs
    if not recommendations:
        recommendations.append("✅ No major issues detected. Model appears well-fitted.")

    # PRINT REPORT
    print(f"\n{'─' * 65}")
    print(f"  FIT DIAGNOSIS: {header}")
    print(f"{'─' * 65}")
    print(f"  Status: {status}")
    print(f"  Epochs trained: {n_epochs}")
    print()
    print(f"  {'Check':<25s}  {'Result':<18s}  Detail")
    print(f"  {'─'*25}  {'─'*18}  {'─'*30}")
    for check_name, (result, detail) in checks.items():
        print(f"  {check_name:<25s}  {result:<18s}  {detail}")
    print(f"\n  Recommendations:")
    for rec in recommendations:
        print(f"    {rec}")
    print(f"{'─' * 65}")

    # DIAGNOSTIC PLOT
    if plot and n_epochs > 1:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'Fit Diagnosis – {header}\nStatus: {status}', fontsize=14, fontweight='bold')
        epochs = range(1, n_epochs + 1)

        ax = axes[0, 0]
        ax.plot(epochs, train_loss, label='Train Loss', color='#2196F3', linewidth=1.5)
        ax.plot(epochs, val_loss, label='Val Loss', color='#F44336', linewidth=1.5)
        ax.fill_between(epochs, train_loss, val_loss, alpha=0.15,
                        color='red' if acc_gap > gap_overfit_threshold else 'gray',
                        label='Gap (overfit zone)' if acc_gap > gap_overfit_threshold else 'Gap')
        if 'best_val_epoch' in details:
            ax.axvline(details['best_val_epoch'], color='green', linestyle='--', alpha=0.7,
                      label=f'Best val (epoch {details["best_val_epoch"]})')
        ax.set_title('Loss Curves')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        ax = axes[0, 1]
        ax.plot(epochs, train_acc, label='Train Acc', color='#2196F3', linewidth=1.5)
        ax.plot(epochs, val_acc, label='Val Acc', color='#F44336', linewidth=1.5)
        ax.plot(epochs, val_f1, label='Val F1', color='#4CAF50', linewidth=1.5, linestyle='--')
        ax.axhline(low_acc_threshold, color='orange', linestyle=':', alpha=0.6,
                   label=f'Underfit threshold ({low_acc_threshold})')
        ax.fill_between(epochs, train_acc, val_acc, alpha=0.1, color='red')
        ax.set_title('Accuracy & F1 Curves')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Score')
        ax.set_ylim([0, 1.05])
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        ax = axes[1, 0]
        loss_gap = val_loss - train_loss
        ax.plot(epochs, loss_gap, label='Loss Gap (val-train)', color='#9C27B0', linewidth=1.5)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
        ax.fill_between(epochs, 0, loss_gap,
                        where=np.array(loss_gap) > 0, alpha=0.2, color='red', label='Overfit zone')
        ax.fill_between(epochs, 0, loss_gap,
                        where=np.array(loss_gap) <= 0, alpha=0.2, color='green', label='Underfit zone')
        ax.set_title('Generalization Gap (Loss)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Val Loss − Train Loss')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        ax = axes[1, 1]
        summary_metrics = {
            'Train Acc': final_train_acc,
            'Val Acc': final_val_acc,
            'Val F1': final_f1,
            'Acc Gap': acc_gap,
            'Loss Ratio': min(loss_ratio, 3.0) / 3.0
        }
        colors_map = []
        for k, v in summary_metrics.items():
            if k == 'Acc Gap':
                colors_map.append('#F44336' if v > gap_overfit_threshold else '#4CAF50')
            elif k == 'Loss Ratio':
                colors_map.append('#F44336' if loss_ratio > loss_divergence_ratio else '#4CAF50')
            elif k in ['Val Acc', 'Val F1']:
                colors_map.append('#F44336' if v < low_acc_threshold else '#4CAF50')
            else:
                colors_map.append('#2196F3')
        bars = ax.barh(list(summary_metrics.keys()), list(summary_metrics.values()),
                       color=colors_map, alpha=0.8, edgecolor='black', linewidth=0.5)
        for bar, val in zip(bars, summary_metrics.values()):
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                   f'{val:.4f}', va='center', fontsize=9)
        ax.set_title('Key Metrics Summary')
        ax.set_xlim([0, 1.2])
        ax.grid(axis='x', alpha=0.3)

        plt.tight_layout()
        plt.show()

    return {
        'status': status,
        'checks': checks,
        'recommendations': recommendations,
        'details': details
    }


# ============================================================
# 23. BATCH DIAGNOSIS
# ============================================================
def diagnose_all_folds(all_results, stages_cfg, plot_per_fold=True, plot_summary=True):
    all_diagnoses = {}

    for stage_name, _, _, class_names in stages_cfg:
        if stage_name not in all_results:
            continue

        fold_results = all_results[stage_name]['fold_results']
        stage_diagnoses = []

        print("\n" + "=" * 70)
        print(f"  FIT DIAGNOSIS REPORT: {stage_name}")
        print("=" * 70)

        for fold_idx, result in enumerate(fold_results):
            diag = diagnose_fit(
                history=result['history'],
                fold_idx=fold_idx,
                stage_name=stage_name,
                plot=plot_per_fold
            )
            stage_diagnoses.append(diag)

        all_diagnoses[stage_name] = stage_diagnoses

        print(f"\n{'═' * 65}")
        print(f"  STAGE SUMMARY: {stage_name}")
        print(f"{'═' * 65}")

        statuses = [d['status'] for d in stage_diagnoses]
        status_counts = defaultdict(int)
        for s in statuses:
            status_counts[s] += 1

        print(f"\n  Fold Status Distribution:")
        for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
            bar = '█' * count
            print(f"    {status:<40s}  {count}/{len(statuses)}  {bar}")

        acc_gaps = [d['details']['acc_gap'] for d in stage_diagnoses]
        val_accs = [d['details']['final_val_acc'] for d in stage_diagnoses]
        val_f1s  = [d['details']['final_val_f1'] for d in stage_diagnoses]

        print(f"\n  Across Folds:")
        print(f"    Acc Gap:  mean={np.mean(acc_gaps):.4f}  std={np.std(acc_gaps):.4f}  "
              f"range=[{np.min(acc_gaps):.4f}, {np.max(acc_gaps):.4f}]")
        print(f"    Val Acc:  mean={np.mean(val_accs):.4f}  std={np.std(val_accs):.4f}")
        print(f"    Val F1:   mean={np.mean(val_f1s):.4f}   std={np.std(val_f1s):.4f}")

        overfit_count = sum(1 for s in statuses if 'OVERFIT' in s.upper())
        underfit_count = sum(1 for s in statuses if 'UNDERFIT' in s.upper())
        good_count = sum(1 for s in statuses if 'GOOD' in s.upper())

        if overfit_count >= 3:
            stage_verdict = '🔴 PREDOMINANTLY OVERFITTING'
        elif underfit_count >= 3:
            stage_verdict = '🔴 PREDOMINANTLY UNDERFITTING'
        elif good_count >= 3:
            stage_verdict = '🟢 PREDOMINANTLY GOOD FIT'
        elif overfit_count >= 2:
            stage_verdict = '🟠 TENDENCY TO OVERFIT'
        elif underfit_count >= 2:
            stage_verdict = '🟠 TENDENCY TO UNDERFIT'
        else:
            stage_verdict = '🟡 MIXED RESULTS'

        print(f"\n  ➤ Stage Verdict: {stage_verdict}")
        print(f"{'═' * 65}")

    if plot_summary:
        _plot_diagnosis_summary(all_diagnoses, stages_cfg)

    return all_diagnoses


def _plot_diagnosis_summary(all_diagnoses, stages_cfg):
    fig, axes = plt.subplots(1, len(all_diagnoses), figsize=(7 * len(all_diagnoses), 5))
    if len(all_diagnoses) == 1:
        axes = [axes]

    color_map = {
        'GOOD': '#4CAF50', 'MILD OVERFIT': '#FFC107', 'OVERFIT': '#FF9800',
        'SEVERE OVERFIT': '#F44336', 'UNDERFIT': '#2196F3',
        'MILD UNDERFIT': '#03A9F4', 'WARNING': '#9E9E9E', 'MIXED': '#9C27B0'
    }

    for idx, (stage_name, _, _, _) in enumerate(stages_cfg):
        if stage_name not in all_diagnoses:
            continue
        diagnoses = all_diagnoses[stage_name]
        ax = axes[idx]

        categories = []
        for d in diagnoses:
            s = d['status'].upper()
            if 'SEVERE' in s:
                categories.append('SEVERE OVERFIT')
            elif 'UNDERFITTING' in s and 'MILD' in s:
                categories.append('MILD UNDERFIT')
            elif 'UNDERFITTING' in s:
                categories.append('UNDERFIT')
            elif 'OVERFITTING' in s and 'MILD' in s:
                categories.append('MILD OVERFIT')
            elif 'OVERFITTING' in s:
                categories.append('OVERFIT')
            elif 'GOOD' in s:
                categories.append('GOOD')
            elif 'WARNING' in s:
                categories.append('WARNING')
            else:
                categories.append('MIXED')

        cat_counts = defaultdict(int)
        for c in categories:
            cat_counts[c] += 1

        labels = list(cat_counts.keys())
        counts = list(cat_counts.values())
        colors = [color_map.get(l, '#9E9E9E') for l in labels]

        ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
        for i, (lbl, cnt) in enumerate(zip(labels, counts)):
            ax.text(i, cnt + 0.05, str(cnt), ha='center', fontweight='bold')
        ax.set_title(stage_name, fontsize=12)
        ax.set_ylabel('Number of Folds')
        ax.set_ylim([0, max(counts) + 1])
        ax.tick_params(axis='x', rotation=30)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Fit Diagnosis Summary Across Folds', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ============================================================
# 24. TRAINING LOOP — METHODOLOGY-COMPLIANT
# ============================================================
# Section 4.4: Class weighting technique
# No label smoothing, no mixup, no warmup, no Conv1D
# ============================================================
def train_lstm_fold(
    fold_data: dict,
    input_size: int = 4,        # Section 4.3.1(a): 4 features
    hidden_size: int = 128,
    num_layers: int = 2,
    num_classes: int = 2,
    dropout: float = 0.3,
    bidirectional: bool = True,
    lr: float = 5e-4,
    weight_decay: float = 1e-4,
    batch_size: int = 32,
    max_epochs: int = 150,
    patience: int = 15,
    class_names: list = None
) -> dict:

    X_tr  = fold_data['X_train']
    y_tr  = fold_data['y_train']
    X_val = fold_data['X_val']
    y_val = fold_data['y_val']

    mask_tr  = create_padding_mask(X_tr)
    mask_val = create_padding_mask(X_val)

    X_tr_t  = torch.FloatTensor(X_tr).to(device)
    y_tr_t  = torch.LongTensor(y_tr).to(device)
    m_tr_t  = torch.FloatTensor(mask_tr).to(device)

    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.LongTensor(y_val).to(device)
    m_val_t = torch.FloatTensor(mask_val).to(device)

    train_ds = TensorDataset(X_tr_t, y_tr_t, m_tr_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # ── Pure LSTM model (no Conv1D) ───────────────────────────
    model = HandwritingLSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        num_classes=num_classes,
        dropout=dropout,
        bidirectional=bidirectional
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Model parameters: {n_params:,}")

    # Section 4.4: "class weighting technique"
    # No label smoothing — plain CrossEntropyLoss with weights
    class_weights = compute_class_weights(y_tr).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-6
    )

    early_stop = EarlyStopping(patience=patience, mode='max')

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'val_f1': [], 'lr': []
    }

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        # No mixup — standard training loop
        for batch_X, batch_y, batch_m in train_loader:
            optimizer.zero_grad()
            logits, _ = model(batch_X, batch_m)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item() * batch_X.size(0)
            preds = logits.argmax(dim=1)
            epoch_correct += (preds == batch_y).sum().item()
            epoch_total += batch_X.size(0)

        train_loss = epoch_loss / epoch_total
        train_acc = epoch_correct / epoch_total

        # ── Validation ────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(X_val_t, m_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_preds = val_logits.argmax(dim=1).cpu().numpy()
            val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()

        val_acc = accuracy_score(y_val, val_preds)
        val_f1  = f1_score(y_val, val_preds, average='weighted')

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        early_stop(val_f1, model)
        if early_stop.early_stop:
            print(f"    Early stop at epoch {epoch + 1}")
            break

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1:>4d} │ "
                  f"loss: {train_loss:.4f}/{val_loss:.4f}  "
                  f"acc: {train_acc:.3f}/{val_acc:.3f}  "
                  f"F1: {val_f1:.3f}  "
                  f"lr: {optimizer.param_groups[0]['lr']:.2e}")

    # ── Load best and evaluate ────────────────────────────────
    early_stop.load_best(model)
    model.eval()
    with torch.no_grad():
        val_logits, val_attn = model(X_val_t, m_val_t)
        val_preds = val_logits.argmax(dim=1).cpu().numpy()
        val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()

    metrics = {
        'accuracy':  accuracy_score(y_val, val_preds),
        'precision': precision_score(y_val, val_preds, average='weighted', zero_division=0),
        'recall':    recall_score(y_val, val_preds, average='weighted', zero_division=0),
        'f1':        f1_score(y_val, val_preds, average='weighted', zero_division=0),
        'confusion_matrix': confusion_matrix(y_val, val_preds),
    }

    try:
        if num_classes == 2:
            metrics['auc'] = roc_auc_score(y_val, val_probs[:, 1])
        else:
            metrics['auc'] = roc_auc_score(y_val, val_probs, multi_class='ovr')
    except ValueError:
        metrics['auc'] = float('nan')

    return {
        'model': model,
        'metrics': metrics,
        'history': history,
        'val_preds': val_preds,
        'val_probs': val_probs,
        'val_true': y_val,
        'best_epoch': len(history['train_loss']) - early_stop.counter
    }


# # ============================================================
# # 25. HYPERPARAMETERS — METHODOLOGY-COMPLIANT
# # ============================================================

# # --- Inspect actual data shape FIRST ---
# sample_fold_data = get_path_a_fold_binary(
#     0, full_df, stages_cfg[0][1], stages_cfg[0][2], SEQUENCE_LENGTH
# )
# ACTUAL_INPUT_SIZE = sample_fold_data['X_train'].shape[-1]
# print(f"  Actual input size from data: {ACTUAL_INPUT_SIZE}")

# LSTM_CONFIG = {
#     'input_size':    ACTUAL_INPUT_SIZE,   # <-- USE ACTUAL SIZE
#     'hidden_size':   32,
#     'num_layers':    1,
#     'dropout':       0.5,
#     'bidirectional': True,
#     'lr':            1e-3,
#     'weight_decay':  1e-3,
#     'batch_size':    16,
#     'max_epochs':    200,
#     'patience':      25,
# }

# print("\n" + "=" * 55)
# print("  LSTM Configuration (Methodology-Compliant)")
# print("=" * 55)
# for k, v in LSTM_CONFIG.items():
#     print(f"  {k:<18s}: {v}")

# print(f"\n  Methodology compliance:")
# print(f"    ✅ input_size = {LSTM_CONFIG['input_size']} (x, y, pressure, in_air)")
# print(f"    ✅ Pure LSTM (no Conv1D front-end)")
# print(f"    ✅ Single-head attention pooling")
# print(f"    ✅ Class weighting (no synthetic oversampling)")
# print(f"    ✅ No label smoothing")
# print(f"    ✅ No mixup augmentation")
# print(f"    ✅ No warmup phase")
# print(f"    ✅ No data augmentation")

# ============================================================
# 25. HYPERPARAMETERS — ENHANCED
# ============================================================

# --- Inspect actual data shape ---
sample_fold_data = get_path_a_fold_binary(
    0, full_df, stages_cfg[0][1], stages_cfg[0][2], SEQUENCE_LENGTH
)
ACTUAL_INPUT_SIZE = sample_fold_data['X_train'].shape[-1]
print(f"  Actual input size from data: {ACTUAL_INPUT_SIZE}")  # Should be 7

LSTM_CONFIG = {
    'input_size':    ACTUAL_INPUT_SIZE,   # 7 now
    'hidden_size':   32,
    'num_layers':    1,
    'dropout':       0.5,
    'bidirectional': True,
    'lr':            1e-3,
    'weight_decay':  1e-3,
    'batch_size':    16,
    'max_epochs':    200,
    'patience':      25,
}

print("\n" + "=" * 55)
print("  LSTM Configuration (Enhanced Features)")
print("=" * 55)
for k, v in LSTM_CONFIG.items():
    print(f"  {k:<18s}: {v}")

print(f"\n  Feature configuration:")
print(f"    Base features:    x, y, pressure, in_air (4)")
print(f"    Derived features: velocity, acceleration, pressure_change (3)")
print(f"    Total:            {ACTUAL_INPUT_SIZE} features per timestep")
print(f"    ✅ All derived features computed from base features only")
print(f"    ✅ No external data introduced")

# ============================================================
# 26. RUN 5-FOLD CV FOR BOTH STAGES
# ============================================================
# Section 4.2: "trained and tested 5 separate times"
# Section 4.4: "class weighting technique"
# ============================================================
all_results = {}

for stage_name, subj_df, label_map, class_names in stages_cfg:
    num_classes = len(class_names)

    print("\n" + "=" * 70)
    print(f"  LSTM – {stage_name}  ({' vs '.join(class_names)})")
    print("=" * 70)

    fold_results = []
    aggregate_metrics = defaultdict(list)

    for fold_idx in range(N_FOLDS):
        print(f"\n  ── Fold {fold_idx + 1}/{N_FOLDS} ──")

        # Section 4.3.1: 4 features, Z-score, clip+pad to L
        # No augmentation
        fold_data = get_path_a_fold_binary(
            fold_idx, full_df, subj_df, label_map, SEQUENCE_LENGTH
        )

        print(f"    Train: {fold_data['X_train'].shape}  "
              f"Val: {fold_data['X_val'].shape}")
        print(f"    Train dist: {dict(zip(*np.unique(fold_data['y_train'], return_counts=True)))}")
        print(f"    Val   dist: {dict(zip(*np.unique(fold_data['y_val'], return_counts=True)))}")

        result = train_lstm_fold(
            fold_data,
            num_classes=num_classes,
            class_names=class_names,
            **LSTM_CONFIG
        )

        fold_results.append(result)

        m = result['metrics']
        print(f"\n    ✅ Best epoch: {result['best_epoch']}")
        print(f"    Accuracy : {m['accuracy']:.4f}")
        print(f"    Precision: {m['precision']:.4f}")
        print(f"    Recall   : {m['recall']:.4f}")
        print(f"    F1       : {m['f1']:.4f}")
        print(f"    AUC      : {m['auc']:.4f}")

        for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
            aggregate_metrics[metric_name].append(m[metric_name])

    # Section 4.5: "reported as the mean and standard deviation"
    print(f"\n{'─' * 70}")
    print(f"  AGGREGATE – {stage_name}")
    print(f"{'─' * 70}")
    print(f"  {'Metric':<12s}  {'Mean':>8s}  {'Std':>8s}  {'Min':>8s}  {'Max':>8s}")
    print(f"  {'─'*12}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}")

    for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        vals = aggregate_metrics[metric_name]
        vals_clean = [v for v in vals if not np.isnan(v)]
        if vals_clean:
            print(f"  {metric_name:<12s}  {np.mean(vals_clean):>8.4f}  "
                  f"{np.std(vals_clean):>8.4f}  "
                  f"{np.min(vals_clean):>8.4f}  {np.max(vals_clean):>8.4f}")

    all_results[stage_name] = {
        'fold_results': fold_results,
        'aggregate': dict(aggregate_metrics)
    }


# ============================================================
# 27. RUN FIT DIAGNOSIS ON ALL RESULTS
# ============================================================
print("\n" + "=" * 70)
print("  RUNNING FIT DIAGNOSIS ON ALL FOLDS")
print("=" * 70)

all_diagnoses = diagnose_all_folds(
    all_results,
    stages_cfg,
    plot_per_fold=True,
    plot_summary=True
)


# ============================================================
# 28. VISUALIZATION – TRAINING CURVES
# ============================================================
def plot_training_curves(all_results, stages_cfg):
    for stage_name, _, _, class_names in stages_cfg:
        if stage_name not in all_results:
            continue

        fold_results = all_results[stage_name]['fold_results']
        n_folds = len(fold_results)

        fig, axes = plt.subplots(2, n_folds, figsize=(5 * n_folds, 8))
        fig.suptitle(f'LSTM Training Curves – {stage_name}', fontsize=14, y=1.02)

        for i, result in enumerate(fold_results):
            h = result['history']
            epochs = range(1, len(h['train_loss']) + 1)

            ax = axes[0, i] if n_folds > 1 else axes[0]
            ax.plot(epochs, h['train_loss'], label='Train', alpha=0.8)
            ax.plot(epochs, h['val_loss'], label='Val', alpha=0.8)
            if result['best_epoch'] is not None:
                ax.axvline(result['best_epoch'], color='red', linestyle='--',
                          alpha=0.5, label=f"Best ({result['best_epoch']})")
            ax.set_title(f'Fold {i+1} – Loss')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

            ax = axes[1, i] if n_folds > 1 else axes[1]
            ax.plot(epochs, h['train_acc'], label='Train', alpha=0.8)
            ax.plot(epochs, h['val_acc'], label='Val', alpha=0.8)
            ax.plot(epochs, h['val_f1'], label='Val F1', alpha=0.8, linestyle='--')
            ax.set_title(f'Fold {i+1} – Accuracy & F1')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Score')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.set_ylim([0, 1.05])

        plt.tight_layout()
        plt.show()

plot_training_curves(all_results, stages_cfg)


# ============================================================
# 29. VISUALIZATION – CONFUSION MATRICES
# ============================================================
def plot_confusion_matrices(all_results, stages_cfg):
    for stage_name, _, _, class_names in stages_cfg:
        if stage_name not in all_results:
            continue

        fold_results = all_results[stage_name]['fold_results']
        n_folds = len(fold_results)

        fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 4))
        fig.suptitle(f'Confusion Matrices – {stage_name}', fontsize=14)

        for i, result in enumerate(fold_results):
            ax = axes[i] if n_folds > 1 else axes
            cm = result['metrics']['confusion_matrix']
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                       xticklabels=class_names, yticklabels=class_names,
                       ax=ax, cbar=False)
            ax.set_title(f'Fold {i+1}\nAcc={result["metrics"]["accuracy"]:.3f}')
            ax.set_ylabel('True')
            ax.set_xlabel('Predicted')

        plt.tight_layout()
        plt.show()

        all_true = np.concatenate([r['val_true'] for r in fold_results])
        all_pred = np.concatenate([r['val_preds'] for r in fold_results])
        cm_agg = confusion_matrix(all_true, all_pred)

        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        sns.heatmap(cm_agg, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names,
                   ax=ax, cbar=True)
        ax.set_title(f'Aggregated CM – {stage_name}')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')
        plt.tight_layout()
        plt.show()

plot_confusion_matrices(all_results, stages_cfg)


# ============================================================
# 30. AGGREGATE METRIC BAR CHART
# ============================================================
def plot_aggregate_bar(all_results, stages_cfg):
    metric_names = ['accuracy', 'precision', 'recall', 'f1', 'auc']

    fig, axes = plt.subplots(1, len(stages_cfg), figsize=(8 * len(stages_cfg), 5))
    if len(stages_cfg) == 1:
        axes = [axes]

    for idx, (stage_name, _, _, class_names) in enumerate(stages_cfg):
        if stage_name not in all_results:
            continue
        agg = all_results[stage_name]['aggregate']
        means = [np.nanmean(agg[m]) for m in metric_names]
        stds  = [np.nanstd(agg[m]) for m in metric_names]

        ax = axes[idx]
        bars = ax.bar(metric_names, means, yerr=stds, capsize=5,
                      color=['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974'],
                      alpha=0.85, edgecolor='black', linewidth=0.5)

        for bar, mean_val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                   f'{mean_val:.3f}', ha='center', va='bottom', fontsize=10)

        ax.set_title(f'{stage_name}', fontsize=13)
        ax.set_ylim([0, 1.15])
        ax.set_ylabel('Score')
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('LSTM – 5-Fold CV Aggregate Metrics', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_aggregate_bar(all_results, stages_cfg)


# ============================================================
# 31. FINAL SUMMARY TABLE
# ============================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY – LSTM (Path A)")
print("=" * 70)

summary_rows = []
for stage_name, _, _, class_names in stages_cfg:
    if stage_name not in all_results:
        continue
    agg = all_results[stage_name]['aggregate']
    row = {'Stage': stage_name}
    for m in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        vals = [v for v in agg[m] if not np.isnan(v)]
        if vals:
            row[f'{m}_mean'] = np.mean(vals)
            row[f'{m}_std']  = np.std(vals)
        else:
            row[f'{m}_mean'] = np.nan
            row[f'{m}_std']  = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

for _, row in summary_df.iterrows():
    print(f"\n{row['Stage']}")
    for m in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        mean_val = row[f'{m}_mean']
        std_val  = row[f'{m}_std']
        print(f"  {m:<12s}: {mean_val:.4f} ± {std_val:.4f}")

print(f"\n{'─' * 70}")
print("FIT DIAGNOSIS VERDICTS:")
print(f"{'─' * 70}")
for stage_name in all_diagnoses:
    diagnoses = all_diagnoses[stage_name]
    for i, d in enumerate(diagnoses):
        print(f"  {stage_name} Fold {i+1}: {d['status']}")

print("\n✅ LSTM training + fit diagnosis complete.")


# ============================================================
# 32. SAVE MODELS
# ============================================================
SAVE_DIR = '/content/drive/MyDrive/teasis/PaHaW/models/'
os.makedirs(SAVE_DIR, exist_ok=True)

for stage_name in all_results:
    fold_results = all_results[stage_name]['fold_results']
    best_fold_idx = np.argmax([r['metrics']['f1'] for r in fold_results])
    best_model = fold_results[best_fold_idx]['model']

    safe_name = stage_name.replace(' ', '_').replace('–', '').replace('__', '_')
    save_path = os.path.join(SAVE_DIR, f'lstm_{safe_name}_best_fold{best_fold_idx+1}.pt')

    diag = all_diagnoses.get(stage_name, [{}] * N_FOLDS)[best_fold_idx]

    # torch.save({
    #     'model_state_dict': best_model.state_dict(),
    #     'config': LSTM_CONFIG,
    #     'metrics': fold_results[best_fold_idx]['metrics'],
    #     'stage_name': stage_name,
    #     'fold_idx': best_fold_idx,
    #     'fit_diagnosis': diag.get('status', 'N/A'),
    #     'preprocessing': {
    #         'features': FEATURES,
    #         'sequence_length': SEQUENCE_LENGTH,
    #     }
    # }, save_path)

    # In Section 32 (SAVE MODELS), update the preprocessing info:
    # Change FEATURES to FEATURES_ENHANCED

    torch.save({
        'model_state_dict': best_model.state_dict(),
        'config': LSTM_CONFIG,
        'metrics': fold_results[best_fold_idx]['metrics'],
        'stage_name': stage_name,
        'fold_idx': best_fold_idx,
        'fit_diagnosis': diag.get('status', 'N/A'),
        'preprocessing': {
            'features': FEATURES_ENHANCED,          # ← updated
            'sequence_length': SEQUENCE_LENGTH,
        }
    }, save_path)

    print(f"Saved: {save_path}")
    print(f"  Best fold: {best_fold_idx + 1}  "
          f"F1: {fold_results[best_fold_idx]['metrics']['f1']:.4f}  "
          f"AUC: {fold_results[best_fold_idx]['metrics']['auc']:.4f}  "
          f"Fit: {diag.get('status', 'N/A')}")

print("\n✅ All models saved.")